In [1]:
import pandas as pd
import numpy as np

In [2]:
def load_observation(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return (
        df.set_axis(df.iloc[0], axis=1)
        .iloc[1:]
        .reset_index(drop=True)
    )

In [3]:
feeding_df = load_observation('../../datasets/observations/Feeding.csv')
feeding_df

,Date,Time,Batch_Id,Feed_Id,Feed Weight (ml),Feed Weight (kg)
0,6/1/2026,7:00:00 AM,1,2,62.5,0.0219
1,6/1/2026,1:00:00 PM,1,2,62.5,0.0219
2,6/1/2026,7:00:00 PM,1,2,62.5,0.0219
3,6/2/2026,7:00:00 AM,1,2,62.5,0.0219
4,6/2/2026,1:00:00 PM,1,2,62.5,0.0219
...,...,...,...,...,...,...
375,11/17/2026,6:00:00 PM,4,17,30,0.0518
376,11/18/2026,7:00:00 AM,4,18,3000,1.8391
377,11/18/2026,7:00:00 AM,4,17,240,0.4145
378,11/18/2026,6:00:00 PM,4,17,NaN,0.0000


In [4]:
sampling_df = load_observation('../../datasets/observations/Sampling.csv')
sampling_df

,Id,Date,Batch_Id,Duration (Days),Age (Days),Sample Size,Sample Weight (kg),Population Size
0,1,7/6/2026,1,35,35,16,0.95,514
1,2,7/13/2026,1,7,42,20,1.9,514
2,3,7/25/2026,1,12,54,18,2.5,512
3,4,8/8/2026,1,14,68,13,2.5,478
4,5,8/15/2026,1,7,75,6,1.8,478
5,6,8/23/2026,1,8,83,384,132.45,477
6,7,8/30/2026,2,3,3,41,0.1,1028
7,8,10/3/2026,2,34,37,55,1.4,1028
8,9,10/17/2026,2,14,51,23,1.65,1028
9,9,11/7/2026,2,21,72,7,1.85,1028


In [5]:
feed_df = load_observation('../../datasets/observations/Feeds.csv')
feed_df

,Id,Date,Brand,Size (mm),Tag (N_S_I),Price,Quantity (Kg),Price Per Kg,Ml to Kg,Used (Kg),Left (Kg)
0,1,6/1/2026,Biomar,2,Feed_1,"₦13,036.00",3.00,"₦4,345.33",0.00056,3.00,0.00
1,2,6/3/2026,Biomar,1.5,Feed_2,"₦7,900.00",1.00,"₦7,900.00",0.00035,1.00,0.00
2,3,6/18/2026,Biomar,2,Feed_3,"₦16,400.00",4.00,"₦4,100.00",0.00055,4.00,0.00
3,4,6/20/2026,Local,2,Feed_4,"₦21,000.00",20.00,"₦1,050.00",0.00058,20.00,0.00
4,5,7/9/2026,Blue Crown,3,Feed_5,"₦25,800.00",14.30,"₦1,804.20",0.00055,14.30,0.00
5,6,7/14/2026,Local,2,Feed_6,"₦21,000.00",20.00,"₦1,050.00",0.00057,20.00,0.00
6,7,7/23/2026,Local,2,Feed_7,"₦43,200.00",40.00,"₦1,080.00",0.00062,40.00,0.00
7,8,8/8/2026,Blue Crown,4,Feed_8,"₦24,300.00",15.00,"₦1,620.00",0.00049,15.00,0.00
8,9,8/11/2026,Ecofloat,4,Feed_9,"₦19,400.00",15.00,"₦1,293.33",0.00047,15.00,0.00
9,10,8/17/2026,New Hope,6,Feed_10,"₦40,000.00",25.00,"₦1,600.00",0.00040,25.00,0.00


### Data Cleaning

In [6]:
feeding_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Date              379 non-null    str  
 1   Time              379 non-null    str  
 2   Batch_Id          379 non-null    str  
 3   Feed_Id           379 non-null    str  
 4   Feed Weight (ml)  377 non-null    str  
 5   Feed Weight (kg)  380 non-null    str  
dtypes: str(6)
memory usage: 29.7 KB


- Parse Date and Time
- Parse Feed Weight (ml and kg)

In [7]:
feeding_df['Date'] = pd.to_datetime(feeding_df['Date']).dt.date
feeding_df['Feed Weight (ml)'] = feeding_df['Feed Weight (ml)'].astype('float')
feeding_df['Feed Weight (kg)'] = feeding_df['Feed Weight (kg)'].astype('float')
feeding_df = feeding_df.sort_values(by='Date')

feeding_df.head()

,Date,Time,Batch_Id,Feed_Id,Feed Weight (ml),Feed Weight (kg)
0,2026-06-01,7:00:00 AM,1,2,62.5,0.0219
1,2026-06-01,1:00:00 PM,1,2,62.5,0.0219
2,2026-06-01,7:00:00 PM,1,2,62.5,0.0219
3,2026-06-02,7:00:00 AM,1,2,62.5,0.0219
4,2026-06-02,1:00:00 PM,1,2,62.5,0.0219


Get the summary of each feed in the feeding dataset

In [30]:
daily_feed_summary = (
    feeding_df.groupby(['Feed_Id', 'Date', 'Batch_Id'], as_index=False)
    .agg(
        feed_weight = ('Feed Weight (kg)', 'sum'),
    )
    .sort_values(by="Date")
)
daily_feed_summary

,Feed_Id,Date,Batch_Id,feed_weight
123,2,2026-06-01,1,0.0657
124,2,2026-06-02,1,0.0657
125,2,2026-06-03,1,0.0657
126,2,2026-06-04,1,0.0867
127,2,2026-06-05,1,0.0867
...,...,...,...,...
120,18,2026-11-17,3,2.4521
121,18,2026-11-17,4,2.4521
110,17,2026-11-17,4,0.2418
122,18,2026-11-18,4,1.8391


In [12]:
sampling_df['Date'] = pd.to_datetime(sampling_df['Date']).dt.date
sampling_df['Sample Weight (kg)'] = sampling_df['Sample Weight (kg)'].astype('float')
sampling_df['Sample Size'] = sampling_df['Sample Size'].astype('float')
sampling_df['Population Size'] = sampling_df['Population Size'].astype('float')

sampling_df['Average Weight (kg)'] = round(sampling_df['Sample Weight (kg)'] / sampling_df['Sample Size'], 3)
sampling_df['Population Weight (kg)'] = round(sampling_df['Population Size'] * sampling_df['Average Weight (kg)'], 3)

sampling_df = sampling_df.sort_values(by='Date', ascending=True)
sampling_df['Weight Change (kg)'] = round(sampling_df.groupby('Batch_Id')['Population Weight (kg)'].diff(), 3)
sampling_df.head()

,Id,Date,Batch_Id,Duration (Days),Age (Days),Sample Size,Sample Weight (kg),Population Size,Average Weight (kg),Population Weight (kg),Weight Change,Weight Change (kg)
0,1,2026-07-06,1,35,35,16.0,0.95,514.0,0.059,30.326,NaN,NaN
1,2,2026-07-13,1,7,42,20.0,1.90,514.0,0.095,48.830,18.504,18.504
2,3,2026-07-25,1,12,54,18.0,2.50,512.0,0.139,71.168,22.338,22.338
3,4,2026-08-08,1,14,68,13.0,2.50,478.0,0.192,91.776,20.608,20.608
4,5,2026-08-15,1,7,75,6.0,1.80,478.0,0.300,143.400,51.624,51.624


In [11]:
feed_df['Date'] = pd.to_datetime(feed_df['Date']).dt.date
feed_df = feed_df.sort_values(by='Date')
feed_df.head()

,Id,Date,Brand,Size (mm),Tag (N_S_I),Price,Quantity (Kg),Price Per Kg,Ml to Kg,Used (Kg),Left (Kg)
0,1,2026-06-01,Biomar,2,Feed_1,"₦13,036.00",3.00,"₦4,345.33",0.00056,3.00,0.00
1,2,2026-06-03,Biomar,1.5,Feed_2,"₦7,900.00",1.00,"₦7,900.00",0.00035,1.00,0.00
2,3,2026-06-18,Biomar,2,Feed_3,"₦16,400.00",4.00,"₦4,100.00",0.00055,4.00,0.00
3,4,2026-06-20,Local,2,Feed_4,"₦21,000.00",20.00,"₦1,050.00",0.00058,20.00,0.00
4,5,2026-07-09,Blue Crown,3,Feed_5,"₦25,800.00",14.30,"₦1,804.20",0.00055,14.30,0.00


In [22]:
s_df = sampling_df[['Id', 'Date', 'Batch_Id', 'Weight Change (kg)']]

In [ ]:
daily_feed_sampling_df = (
    pd.merge_ordered(daily_feed_summary, s_df, on=['Date', 'Batch_Id'], fill_method="ffill", suffixes=("_feed", "_sample"))
    .dropna()
)
daily_feed_sampling_df

,Feed_Id,Date,Batch_Id,feed_weight,Id,Weight Change (kg)
0,2,2026-06-01,1,0.0657,NaN,NaN
1,2,2026-06-02,1,0.0657,NaN,NaN
2,2,2026-06-03,1,0.0657,NaN,NaN
3,2,2026-06-04,1,0.0867,NaN,NaN
4,2,2026-06-05,1,0.0867,NaN,NaN
...,...,...,...,...,...,...
206,17,NaT,NaN,0.4145,NaN,NaN
207,17,NaT,NaN,0.4145,NaN,NaN
208,17,NaT,NaN,0.4145,NaN,NaN
209,17,NaT,NaN,0.4145,NaN,NaN


In [29]:

feed_summary = (
    daily_feed_sampling_df.groupby(["Feed_Id", "Id"], as_index=False)
    .apply(lambda x: pd.Series({
        'total_feed_weight': x['feed_weight'].sum(),
        # 'weight_gained': x['Weight Change (kg)'][0] if 
    }))
)

feed_summary

,Feed_Id,Id,total_feed_weight
0,10,5,21.7956
1,10,6,8.0353
2,10,9,3.2041
3,11,6,0.0736
4,11,7,4.9266
5,12,6,0.1843
6,12,7,0.8155
7,13,7,23.1816
8,13,8,16.8180
9,14,8,9.6210


In [20]:
feed_summary['FCR'] = round(feed_summary['feed_weight'] / feed_summary['population_weight_gained'], 3)
feed_summary

,Feed_Id,Id,feed_weight,population_weight_gained,FCR
0,10,5,21.7956,51.624,0.422
1,10,6,8.0353,21.165,0.380
2,10,9,3.2041,48.316,0.066
3,11,6,0.0736,21.165,0.003
4,11,7,4.9266,NaN,NaN
5,12,6,0.1843,21.165,0.009
6,12,7,0.8155,NaN,NaN
7,13,7,23.1816,NaN,NaN
8,13,8,16.8180,23.644,0.711
9,14,8,9.6210,23.644,0.407
